# Vertical integration: RNA + ATAC

Vertical integration combines modalities measured in the same cells. This tutorial is for RNA and ATAC, as in 10x Multiome. It runs scMM and VIMCCA on `D27mini_vertical`, 5,000 cells of the benchmark dataset `D27`, and then on data in your own format.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_vertical_rna_atac.ipynb).

## 1. Install

This cell installs `multibench-sc`.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.0" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D27mini_vertical` (24 MB) once. `mtb.env.install` downloads the environments for scMM and VIMCCA. The download is 1.2 GB.

In [ ]:
import multibench as mtb

mtb.data.fetch("D27mini_vertical")
METHODS = ["scMM", "VIMCCA"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D27mini_vertical` and scores its output with the scIB metrics. The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D27mini_vertical")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D27mini_vertical", "vertical", methods=METHODS,
                  out_dir="out/D27mini_vertical")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. The bubbles and Overall bars rank the rows within each column, so a small gap can look large: `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts, an ATAC matrix of the same cells and a cell type for each cell, as two `.h5ad` files.

Starting from something else? See the recipes for [a 10x `.h5` file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-10x-h5), [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv), [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels) and [several donors or samples](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-donors).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:atac"` the matrix `obsm["atac"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

A 10x Multiome file read with scanpy and `gex_only=False` holds genes and peaks in one `X`. The recipe for a 10x `.h5` file splits it into the two objects of this section. Without a split or a filter, `export_dataset` raises rather than write genes and peaks into one file.

A 10x Multiome MuData goes in with one call. Here the labels are in `mdata.obs`. For labels in `mdata["rna"].obs`, write `labels="rna:celltype"`.

```python
mtb.io.export_dataset(mdata, "mydata/" + NAME, rna="rna", atac="atac",
                      atac_kind="peak", labels="obs:celltype",
                      category="vertical")
```

Without `methods=`, `scan` lists every method that reads the modalities in the folder.

</details>

So that the notebook runs as it is, the next cell writes two small example files, `my_rna.h5ad` and `my_atac.h5ad`, from 60% of the cells of `D27mini_vertical`.

In [ ]:
mtb.data.write_example("D27mini_vertical")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna = ad.read_h5ad("my_rna.h5ad")     # <- your RNA file
atac = ad.read_h5ad("my_atac.h5ad")   # <- your ATAC file, the same cells
print(rna)
print(atac)

Write the dataset folder. Change each name in quotes marked `# <-`. The printout above lists the names in your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYMULTIOME"   # <- a new name for your own data
mtb.io.export_dataset(rna, "mydata/" + NAME,
                      rna="X",                # <- where the RNA counts are:
                                              #    "X" or "layer:counts"
                      atac=atac,              # the ATAC file, peak counts in X
                      atac_kind="peak",       # the methods here read peaks
                      labels="obs:celltype",  # <- the cell-type column
                      overwrite=True)
mtb.scan("mydata/" + NAME, "vertical", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "vertical", methods=METHODS,
                   out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

14 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_vertical_rna_atac_all.ipynb) runs them all. It downloads 13 envs, 31.8 GB on a computer without a GPU and 37.6 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.